<a href="https://colab.research.google.com/github/epears04/DATA-4620-HW3/blob/main/DATA_4620_HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import torch
import os
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm, trange

from sklearn.model_selection import train_test_split

In [ ]:
device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.available() else 'cpu'

In [ ]:
!wget -q -nc -O facial_keypoints.npz "https://www.dropbox.com/scl/fi/27qggijmythfjg04s24xq/facial_keypoints.npz?rlkey=h91gwodhrfuz8hrc7ux9qnq7s&dl=1"

In [ ]:
data = np.load('facial_keypoints.npz')
images = data['images']
keypoints = data['keypoints']

In [ ]:
def masked_mae_loss(y_pred: torch.Tensor, y_true: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """ Compute masked mean absolute error (MAE) loss, ignoring NaN values.
        This function returns the numerator and denominator separately;
        the loss is calculated as num/den.

        To calculate the average loss over batches, you should
        separately sum the numerators and demoninators,
        and then divide: total_num / total_den.

        Arguments:
            y_pred: predicted keypoints [B,C]
            y_true: ground truth keypoints (may contain NaNs) [B,C]

        Returns:
            numerator, denominator of MAE
    """
    mask = 1-torch.isnan(y_true).float()
    diff = torch.abs(y_true-y_pred)
    return torch.nansum(diff*mask), torch.nansum(mask)